# UTKFace EDA — Phase 2

Quick look at the dataset before any training. Goal: understand age distribution, demographic balance, and label noise so the bias audit later is not a surprise.

Run this from the repo root after `data/utkface/` is populated.

In [ ]:
# imports + path setup — works whether the notebook is opened from notebooks/ or repo root
import sys
from pathlib import Path

# add repo root to sys.path so `import src.*` works
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import AGE_BUCKETS, ETHNICITY_LABELS, GENDER_LABELS, UTKFACE_DIR
from src.data.utkface_dataset import UTKFaceDataset

sns.set_theme(style='whitegrid')
print(f'looking for images in: {UTKFACE_DIR}')
print(f'folder exists: {UTKFACE_DIR.exists()}')

## 1. Load the dataset

No transforms here — just want labels. The dataset class also tells us how many filenames were too noisy to parse.

In [ ]:
ds = UTKFaceDataset(transform=None)
print(f'valid samples loaded: {len(ds):,}')
print(f'noisy filenames skipped: {ds.skipped_count:,}')

# pull labels into a DataFrame for plotting
labels = ds.get_label_arrays()
df = pd.DataFrame(labels)
df['gender_name'] = df['gender'].map(GENDER_LABELS)
df['ethnicity_name'] = df['ethnicity'].map(ETHNICITY_LABELS)
df.head()

## 2. Age distribution

Looking for skew. If most samples are 20-40, the model will be best there and weakest at the extremes (children, elderly). That gap shows up in the bias audit later.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df['age'], bins=50, ax=ax)
ax.set_title('UTKFace — age distribution')
ax.set_xlabel('age')
ax.set_ylabel('count')
plt.tight_layout()
plt.show()

print('summary stats:')
print(df['age'].describe().round(1))

In [ ]:
# bucketed view — easier to read for the bias report later
def to_bucket(age):
    for lo, hi in AGE_BUCKETS:
        if lo <= age <= hi:
            return f'{lo}-{hi}'
    return 'out_of_range'

df['age_bucket'] = df['age'].apply(to_bucket)
bucket_order = [f'{lo}-{hi}' for lo, hi in AGE_BUCKETS]

fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, x='age_bucket', order=bucket_order, ax=ax)
ax.set_title('age buckets — used by the bias audit')
plt.tight_layout()
plt.show()

df['age_bucket'].value_counts().reindex(bucket_order)

## 3. Gender and ethnicity balance

Imbalanced groups predict where the model will under-perform. The bias audit in Phase 5 will check this directly — this is just early visibility.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x='gender_name', ax=axes[0])
axes[0].set_title('gender balance')

sns.countplot(data=df, x='ethnicity_name', ax=axes[1],
              order=[ETHNICITY_LABELS[i] for i in range(5)])
axes[1].set_title('ethnicity balance')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

print('gender counts:')
print(df['gender_name'].value_counts())
print('\nethnicity counts:')
print(df['ethnicity_name'].value_counts())

## 4. Cross-tabs — where is the data thinnest?

Per-group sample counts. Cells with under ~100 samples are noisy enough that the model will not learn that subgroup well. Useful to flag now.

In [ ]:
ct_gender_age = pd.crosstab(df['gender_name'], df['age_bucket'])[bucket_order]
ct_ethnicity_age = pd.crosstab(df['ethnicity_name'], df['age_bucket'])[bucket_order]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.heatmap(ct_gender_age, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('gender x age bucket')
sns.heatmap(ct_ethnicity_age, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('ethnicity x age bucket')
plt.tight_layout()
plt.show()

## 5. Sample image grid

Visual sanity check — do the labels actually match the faces? Spotting label noise early saves debugging time later.

In [ ]:
from PIL import Image

rng = np.random.default_rng(42)
idxs = rng.choice(len(ds), size=12, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, i in zip(axes.flatten(), idxs):
    item = ds[int(i)]
    img = Image.open(item['path']).convert('RGB')
    ax.imshow(img)
    age = item['age'].item()
    g = GENDER_LABELS[item['gender'].item()]
    e = ETHNICITY_LABELS[item['ethnicity'].item()]
    ax.set_title(f'{age}y / {g} / {e}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Findings — what to remember for training

Fill this in after running the cells above. Keep it short — three bullets max:

- age distribution: ...
- demographic balance: ...
- label noise spotted: ...

These notes feed directly into the Phase 5 bias audit and the README's 'Why I Built It' section.